In [1]:
import cobra
import itertools
import uuid
import numpy as np

import sys
sys.path.insert(1, '../../scripts/')
from macromolecules.macromolecule import Macromolecule
from utils import machinery as mach

No objective coefficients in model. Unclear what should be optimized


In [12]:
cotransloc_ids = set([mid + '_folded_protein_r' for mid in mach.ctnm + mach.translation_efs]) 

True

In [2]:
def flatten_list(list_):
    return [item for sublist in list_ for item in sublist]
    
class Complex(Macromolecule):
    def __init__(self, metabolites, complex_id = None, ignore_compartment = False):
        '''
        Inputs:
        metabolites is a list of Macromolecule objects (protein or RNA or complex, not generic metabolites)
        complex_id is a string for the id of the complex metabolite, otherwise will form a random id
        ignore_compartment is a boolean whether ot ignore the metabolite compartments, mainly for internal use
        Output:
        A Macromolecule object representing the complex formed between macromolecules
        
        '''
        # checks
        if type(metabolites) != list or len(metabolites) == 0:
            raise ValueError('Must provide a list of macromolecules to form complex')
        # cobra metabolite not set up, check for bc they don't have the attribute .type
        if len([m for m in metabolites if not(isinstance(m, Macromolecule))]) > 0:
            raise ValueError('Generic cobra.Metabolite cannot form complexes with macromolecules currently')
        
        self.type = 'complex'
        self.components = {m: metabolites.count(m) for m in metabolites}
        # parse compartment    
        compartments = list(set([m.compartment for m in self.components]))
        
        # test compartment consistency - exception of ribosome complexes
        comp_ids = [m.id for m in self.components]
        cotransloc_cond = (len(cotransloc_ids.difference(comp_ids))==0) or ('mature_ribosome_complex_c' in comp_ids)
        
        if len(compartments) == 1:
            compartment = compartments[0]
        elif (sorted(compartments) == ['c', 'r']) and cotransloc_cond:
            compartment = 'c'
        else:
            raise ValueError('Metabolites forming a complex must all be in the same compartment')
        
        
        # parse metabolite id
        if complex_id == None:
            self.temp_id = str(uuid.uuid4().fields[0])
        else: 
            self.temp_id = complex_id
        
        elements = dict()
        for m,count in self.components.items():
            for k,v in m.elements.items():
                if k in elements.keys():
                    elements[k] += v*count
                else:
                    elements[k] = v*count
        
        # make the metabolite
        Macromolecule.__init__(self, id = self.temp_id + '_complex_' + compartment, compartment = compartment,
                                  charge = sum([m.charge*count for m, count in self.components.items()]), 
                              elements = elements)
        
#         self.add_alpha_p()                         
        self.reaction_id = None # none before running form_complex(); this is used in update_id() method
    
    def update_id(self, new_id = None):
        '''In cases where complex id is too long (see build_me_model generate_complex_reactions method)'''

        if self.reaction_id is not None:
            raise ValueError('Reaction and complex IDs will be consistent since you are updating the id after forming the reaction.')
        if new_id is None:
            self.temp_id = str(uuid.uuid4().fields[0])
        else:
            self.temp_id = new_id
        self.id = self.temp_id + '_complex_' + self.compartment
    
    
    def form_complex(self, reaction_id = None):

        '''
        Output: A cobra.Reaction object representing the complex formation between metabolites stored in self.complex_formation

        '''
        
        if reaction_id is None:
            self.reaction_id = self.temp_id + '_COMPLEX_FORMATION' + self.compartment
        else:
            self.reaction_id = reaction_id + '_COMPLEX_FORMATION' + self.compartment        
        
        
        complex_formation = cobra.Reaction(self.reaction_id)
        rxn = {m: -count for m,count in self.components.items()}
        rxn[self] = 1
        complex_formation.add_metabolites(rxn)
        complex_formation.lower_bound = -1000 # reversible
        
        return complex_formation
    
    def decompose_complex(self, decomposed_complex = None):
        '''Recursive method to get the complex by its individual components, including nested complexes'''
        if decomposed_complex == None:
            all_metab = flatten_list([[m]*count for m, count in self.components.items()])
            decomposed_complex = Complex(metabolites = all_metab, complex_id = 'ignore')
        
        if 'complex' not in [m.type for m in decomposed_complex.components.keys()]:
            return decomposed_complex.components
        else:
            metabolites_ = flatten_list([[m]*count for m, count in decomposed_complex.components.items() if m.type != 'complex'])
            metabolites_ += flatten_list(flatten_list([[[m_]*count_ for m_, count_ in m.components.items()]*count for m, count in decomposed_complex.components.items() if m.type == 'complex']))
            return self.decompose_complex(decomposed_complex = Complex(metabolites = metabolites_, complex_id = 'ignore'))

    def get_complex_biomass(self):
        '''Returns a dictionary of the complex biomass by its individual component types'''

        biomass_by_type = dict()
        for m, count in self.decompose_complex().items():
            if m.type in biomass_by_type.keys():
                biomass_by_type[m.type] += count*(m.formula_weight/1000)
            else:
                biomass_by_type[m.type] = count*(m.formula_weight/1000)

        return biomass_by_type
#     def add_alpha_p(self):
#         self.alpha_p = np.median([p.alpha_p for p in self.decompose_complex() if isinstance(p, Protein)])# and p.alpha_p is not None])
#         if pd.isna(self.alpha_p):
#             self.alpha_p = None